In [13]:
import os 
import sys
import json
import pickle
from pathlib import Path

from tqdm import tqdm 
import numpy as np 
import torch
import torch.nn.functional as f
from torch.utils.data import dataset, dataloader
from transformers import AutoTokenizer, AutoModel
import datasets
from datasets import load_dataset
from PIL import Image
import data_utils

In [14]:
data_path = Path('/data/haozhen/uouo/mosaic_output/model-cascade-small-vlm-shuffle-rag')
if data_path not in sys.path:
    sys.path.append(data_path)

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{data_path}"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [16]:
# with open(os.path.join(data_path, 'gpt-4o_uouo_desc.pkl'), 'rb') as file:
#     uouo_query_list = pickle.load(file)

with open(os.path.join(data_path, 'uouo_test_queries.pkl'), 'rb') as file:
    uouo_query_list = pickle.load(file)

pil_img_list = []
for d in uouo_query_list:
    img_path = d['img_path']
    image = Image.open(img_path)
    pil_img_list.append(image)

test_dataset_single_image = datasets.Dataset.from_dict({'image_1': pil_img_list})

In [ ]:
# # load mmmu dataset
# validation_dataset = load_dataset("lmms-lab/MMMU", split="validation")
# dev_dataset = load_dataset("lmms-lab/MMMU", split="dev")
# test_dataset = load_dataset("lmms-lab/MMMU", split="test")

# # load saved conversation with gpt and attach to dataset
# gpt_conversation_path = data_path / 'gpt_desc/mmmu_val_gpt4o_response_v2.jsonl' #TODO: actually the test set
# gpt_conversations = []
# with open(gpt_conversation_path, 'r') as f:
#     for line in f:
#         # each line is a json
#         gpt_conversations.append(line.strip().strip('"'))
# data_conversation = datasets.Dataset.from_dict({"conversations": gpt_conversations})
# test_dataset = datasets.concatenate_datasets([test_dataset, data_conversation], axis=1)

# # # load selected hashtags
# # selected_hashtags_path = data_path / 'embeddings/single_keyword_embeddings_dict_gte-base-en-v1.5.pkl'
# # with open(selected_hashtags_path, 'rb') as f:
# #     selected_hashtags = pickle.load(f)
# # print(len(selected_hashtags[0]))
# # print(len(selected_hashtags[1]))
# # print(len(selected_hashtags[2]))

# # load keywords
# keyword_dir = data_path / 'test_keyword'
# keyword_test_list = data_utils.get_extracted_keywords(keyword_dir)
# data_keyword = datasets.Dataset.from_dict({"keywords": keyword_test_list})
# print(len(data_keyword))
# print(len(test_dataset))
# # test_dataset = datasets.concatenate_datasets([test_dataset, data_keyword], axis=1)

# # only keep single image questions
# test_dataset_single_image = test_dataset.filter(lambda x: x['image_2'] is None)
# print(len(test_dataset_single_image))

10547
10500
9702


In [8]:
# test_dataset[0]

In [17]:
import numpy as np
import torch

print("Torch version:", torch.__version__)


import clip
clip.available_models()

import os
from PIL import Image
import numpy as np
import torch

from collections import defaultdict
import numpy as np
import pickle
from tqdm import tqdm
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

device = 'cuda:1'
model, preprocess = clip.load("ViT-B/32")
model.to(device).eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

print(preprocess)
def mmmu_clip_preprocess_collate_fn(batch):
    images = torch.stack([preprocess(sample['image_1'].convert('RGB')) for sample in batch])
    return {'image_1': images}
dataloader = torch.utils.data.DataLoader(test_dataset_single_image, 
                                         batch_size=64,
                                         collate_fn=mmmu_clip_preprocess_collate_fn)

print(f'num of data: {len(dataloader)}')
# image = Image.open(img_path).convert('RGB')
# image_input = preprocess(image).unsqueeze(0).cuda()
# with torch.no_grad():
#     image_features = model.encode_image(image_input).float()

clip_features = []
print("here")
with torch.no_grad():
    for batch in tqdm(dataloader):
        images = batch['image_1'].to(device)
        # text_inputs = clip.tokenize(batch['conversation'], truncate=True).to('cuda')
        image_features = model.encode_image(images).float()
        # text_features = model.encode_text(text_inputs).float()
        #logits_per_image, logits_per_text = model(images.to('cuda'), text_inputs)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(image_features)
        #print(probs)
        #text_features = model.encode_text(labels)
        #logits_per_image, logits_per_text = model(image, text)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(probs)
        #clip_features.append[[image_features,text_features]]
        clip_features.append(image_features.cpu())
print(len(clip_features))
print(clip_features[0].shape)
#filename = 'mean_features.pkl'
#with open(filename, 'wb') as file:
#    pickle.dump(mean_features, file)

Torch version: 2.5.1+cu124
Model parameters: 151,277,313
Input resolution: 224
Context length: 77
Vocab size: 49408
Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_image_to_rgb at 0x7fca8c3ac9d0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
num of data: 7
here


100%|██████████| 7/7 [00:09<00:00,  1.41s/it]

7
torch.Size([64, 512])


In [18]:
clip_features_concat = torch.cat(clip_features, dim=0)
os.mkdir(os.path.join(data_path, 'clip'))
clip_embeddings_file = data_path / 'clip/uouo_test_image_clip_embd_cold_start.pkl'
with open(clip_embeddings_file, 'wb') as f:
    pickle.dump(clip_features_concat, f)

In [12]:
clip_features_concat.shape

torch.Size([409, 512])

In [28]:
val_clip_embeddings_file = '/home/jizej/Workspaces/cache-of-thoughts/data/mmmu/val/clip/cold_start_single_image.pkl'
with open(val_clip_embeddings_file, 'rb') as f:
    val_clip_features = pickle.load(f)

In [29]:
val_clip_features = np.array(val_clip_features)
val_clip_features = val_clip_features.squeeze(2)
print(val_clip_features.shape)
val_clip_features = torch.tensor(val_clip_features)

(857, 2, 512)


In [31]:
val_clip_image_features = val_clip_features[:, 0, :]

In [36]:
val_clip_image_features.shape

torch.Size([857, 512])

In [37]:
dump_to = '/home/jizej/Workspaces/cache-of-thoughts/data/mmmu/val/clip/mmmu_val_image_clip_embd_cold_start.pkl'
with open(dump_to, 'wb') as f:
    pickle.dump(val_clip_image_features, f)

In [34]:
# read from dump to
with open(dump_to, 'rb') as f:
    clip_features = pickle.load(f)

In [35]:
clip_features.shape

torch.Size([857, 2, 512])